# **Project Overview**

This project focuses on predicting loan defaults using customer application data and historical credit behavior. The workflow includes data cleaning, feature aggregation, feature selection, machine learning model comparison, hyperparameter tuning, and threshold optimization. The final goal is to identify customers who are more likely to default while prioritizing effective default detection.


In [79]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix,
    classification_report
)


from sklearn.model_selection import RandomizedSearchCV
import joblib

## MERGING ALL THE 7 FILES

In [2]:
# main_df = pd.read_csv('cleaned_application_train.csv')

In [3]:
# bureau = pd.read_csv('bureau_aggregated.csv')
# bureau_balance = pd.read_csv('bureau_balance_aggregated.csv')
# previous_app = pd.read_csv('previous_application_aggregated.csv')
# pos_cash = pd.read_csv('pos_cash_aggreagate.csv')
# credit_card = pd.read_csv('credit_card_aggregated.csv')
# installments = pd.read_csv('installments_aggregated.csv')

In [4]:
# print(bureau.columns.tolist())
# print(bureau_balance.columns.tolist())
# print(previous_app.columns.tolist())
# print(pos_cash.columns.tolist())
# print(credit_card.columns.tolist())
# print(installments.columns.tolist())

In [5]:
# final_df = (
#     main_df
#     .merge(bureau, on='SK_ID_CURR', how='left')
#     .merge(bureau_balance, on='SK_ID_CURR', how='left')
#     .merge(previous_app, on='SK_ID_CURR', how='left')
#     .merge(pos_cash, on='SK_ID_CURR', how='left')
#     .merge(credit_card, on='SK_ID_CURR', how='left')
#     .merge(installments, on='SK_ID_CURR', how='left')
# )

In [6]:
# print("Shape:", final_df.shape)

# print(
#     "Duplicate customers:",
#     final_df['SK_ID_CURR'].duplicated().sum()
# )

# print(
#     "Unique customers:",
#     final_df['SK_ID_CURR'].nunique()
# )

In [7]:
# # SAVING THE FINAL FILE
# final_df.to_csv(
#     'final_modeling_dataset.csv',
#     index=False
# )

### LOADING THE FINAL FILE

In [8]:
df = pd.read_csv('final_modeling_dataset.csv')

In [9]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [10]:
df.shape

(307507, 262)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307507 entries, 0 to 307506
Columns: 262 entries, SK_ID_CURR to INSTALL_PAYMENT_COMPLETION_RATIO
dtypes: float64(207), int64(39), object(16)
memory usage: 614.7+ MB


In [12]:
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,AGE,YEARS_EMPLOYED,INCOME_PER_FAMILY_MEMBER,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,ANNUITY_CREDIT_RATIO,BUREAU_TOTAL_LOANS,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_DAYS_CREDIT_MEAN,BUREAU_OVERDUE_DAYS_MAX,BUREAU_OVERDUE_DAYS_MEAN,BUREAU_TOTAL_CREDIT,BUREAU_AVG_CREDIT,BUREAU_MAX_CREDIT,BUREAU_TOTAL_DEBT,BUREAU_AVG_DEBT,BUREAU_MAX_DEBT,BUREAU_TOTAL_OVERDUE,BUREAU_MAX_OVERDUE,BUREAU_TOTAL_CREDIT_PROLONG,BUREAU_MAX_CREDIT_PROLONG,BUREAU_MAX_OVERDUE_RECORD_COUNT,BUREAU_ANNUITY_RECORD_COUNT,BUREAU_ACTIVE_LOANS,BUREAU_ACTIVE_TOTAL_CREDIT,BUREAU_ACTIVE_AVG_CREDIT,BUREAU_ACTIVE_TOTAL_DEBT,BUREAU_ACTIVE_AVG_DEBT,BUREAU_ACTIVE_OVERDUE_DAYS_MAX,BUREAU_ACTIVE_OVERDUE_DAYS_MEAN,BUREAU_CREDIT_TYPE_CAR_MORTGAGE_COUNT,BUREAU_CREDIT_TYPE_CONSUMER_COUNT,BUREAU_CREDIT_TYPE_CREDIT_CARD_COUNT,BUREAU_CREDIT_TYPE_OTHER_COUNT,BUREAU_BALANCE_TOTAL_RECORDS,BUREAU_BALANCE_TOTAL_DPD_MONTHS,BUREAU_BALANCE_UNIQUE_CREDITS,BUREAU_BALANCE_MIN_MONTH,BUREAU_BALANCE_MAX_MONTH,BUREAU_BALANCE_DPD_RATIO,PREV_APP_TOTAL_APPLICATIONS,PREV_APP_TOTAL_APPLICATION_AMOUNT,PREV_APP_AVG_APPLICATION_AMOUNT,PREV_APP_MAX_APPLICATION_AMOUNT,PREV_APP_TOTAL_CREDIT,PREV_APP_AVG_CREDIT,PREV_APP_MAX_CREDIT,PREV_APP_AVG_ANNUITY,PREV_APP_MAX_ANNUITY,PREV_APP_AVG_DOWN_PAYMENT,PREV_APP_MAX_DOWN_PAYMENT,PREV_APP_AVG_GOODS_PRICE,PREV_APP_MAX_GOODS_PRICE,PREV_APP_AVG_PAYMENT_COUNT,PREV_APP_MAX_PAYMENT_COUNT,PREV_APP_AVG_DOWN_PAYMENT_RATE,PREV_APP_MAX_DOWN_PAYMENT_RATE,PREV_APP_OLDEST_DECISION,PREV_APP_MOST_RECENT_DECISION,PREV_APP_AVG_DECISION_TIME,PREV_APP_STATUS_APPROVED_COUNT,PREV_APP_STATUS_CANCELED_COUNT,PREV_APP_STATUS_REFUSED_COUNT,PREV_APP_STATUS_UNUSED_OFFER_COUNT,PREV_APP_APPROVAL_RATE,PREV_APP_REFUSAL_RATE,PREV_APP_CONTRACT_CASH_LOANS_COUNT,PREV_APP_CONTRACT_CONSUMER_LOANS_COUNT,PREV_APP_CONTRACT_REVOLVING_LOANS_COUNT,PREV_APP_CLIEN

### **Light validation**

In [13]:
print("Shape:", df.shape)
print("-"*100)
print("Duplicate rows:", df.duplicated().sum())
print("-"*100)
print("Duplicate SK_ID_CURR:", df['SK_ID_CURR'].duplicated().sum())
print("-"*100)
print("Target distribution:")
print("-"*100)
print(df['TARGET'].value_counts())
print("-"*100)
print(df['TARGET'].value_counts(normalize=True))

Shape: (307507, 262)
----------------------------------------------------------------------------------------------------
Duplicate rows: 0
----------------------------------------------------------------------------------------------------
Duplicate SK_ID_CURR: 0
----------------------------------------------------------------------------------------------------
Target distribution:
----------------------------------------------------------------------------------------------------
TARGET
0    282682
1     24825
Name: count, dtype: int64
----------------------------------------------------------------------------------------------------
TARGET
0    0.91927
1    0.08073
Name: proportion, dtype: float64


**MISSING VALUES**

In [14]:
df.isna().sum().sort_values(ascending=False).head()

,0
CC_AVG_CARD_DPD_RATE,220602
CC_TOTAL_CARD_DPD_MONTHS,220602
CC_AVG_CARD_DPD,220602
CC_MAX_CARD_DPD,220602
CC_MAX_MATURED_INSTALLMENTS,220602


In [15]:
missing_pct = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_pct.head(20)

,0
CC_AVG_CARD_DPD_RATE,71.738855
CC_TOTAL_CARD_DPD_MONTHS,71.738855
CC_AVG_CARD_DPD,71.738855
CC_MAX_CARD_DPD,71.738855
CC_MAX_MATURED_INSTALLMENTS,71.738855
CC_TOTAL_OTHER_DRAWING_COUNT,71.738855
CC_TOTAL_POS_DRAWING_COUNT,71.738855
CC_TOTAL_ATM_DRAWING_COUNT,71.738855
CC_TOTAL_DRAWING_COUNT,71.738855
CC_TOTAL_OTHER_DRAWINGS,71.738855


In [16]:
# check constant columns
X_temp = df.drop(columns=['SK_ID_CURR', 'TARGET'])

constant_cols = [
    col for col in X_temp.columns
    if X_temp[col].nunique(dropna=False) <= 1
]

print("Constant columns:", constant_cols)
print("Number of constant columns:", len(constant_cols))

Constant columns: []
Number of constant columns: 0


### **Separate features and target**

In [17]:
X = df.drop(columns=['SK_ID_CURR', 'TARGET'])
y = df['TARGET']

### **Train / Val / Test Split**
* 70% / 15% / 15%.

In [18]:
# First: separate 15% test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Second: take 15% of the original data as validation
# Since X_temp is 85% of the original data:
# 15 / 85 = 0.17647
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=15/85,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape, "Default rate:", y_train.mean())
print("Val:  ", X_val.shape,   "Default rate:", y_val.mean())
print("Test: ", X_test.shape,  "Default rate:", y_test.mean())

Train: (215254, 260) Default rate: 0.08072788426695904
Val:   (46126, 260) Default rate: 0.08073537701079651
Test:  (46127, 260) Default rate: 0.08073362672621241


In [19]:
# Numerical & Categorical columns

num_features = X_train.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

cat_features = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

print("Numerical features:", len(num_features))
print("Categorical features:", len(cat_features))

Numerical features: 244
Categorical features: 16


In [20]:
# Preprocessing Pipeline

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median'))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(
            handle_unknown='ignore',
            drop='first',
            sparse_output=False
        ))
    ]
)

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
])

For this baseline, median imputation is fine. We can make the historical missingness treatment more sophisticated after we see baseline performance.

### **Baseline Models**

In [21]:
# Define the four models

models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    ),

    'XGBoost': XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1
    ),

    'LightGBM': LGBMClassifier(
        n_estimators=100,
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    )
}

In [22]:
trained_models = {}

results = []

for name, model in models.items():

    print(f"Training {name}...")

    # Create pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Train
    pipeline.fit(X_train, y_train)

    trained_models[name] = pipeline

    # Predictions
    y_pred = pipeline.predict(X_val)
    y_prob = pipeline.predict_proba(X_val)[:, 1]

    # Evaluation
    results.append({
        'Model': name,
        'ROC-AUC': roc_auc_score(y_val, y_prob),
        'Recall': recall_score(y_val, y_pred),
        'Precision': precision_score(y_val, y_pred),
        'F1-Score': f1_score(y_val, y_pred)
    })

results_df = (
    pd.DataFrame(results)
    .sort_values('ROC-AUC', ascending=False)
    .reset_index(drop=True)
)

display(results_df)

Training Random Forest...
Training XGBoost...
Training LightGBM...


,Model,ROC-AUC,Recall,Precision,F1-Score
0,LightGBM,0.784547,0.040279,0.555556,0.075113
1,XGBoost,0.769887,0.066058,0.470363,0.115846
2,Random Forest,0.741291,0.002148,0.363636,0.004271


### **Feature Gain**

**Step 1 — Get the trained LightGBM pipeline**

In [23]:

lightgbm_pipeline = trained_models['LightGBM']

lightgbm_model = lightgbm_pipeline['model']
preprocessor_fitted = lightgbm_pipeline['preprocessor']

**Step 2 — Get the transformed feature names**

In [24]:

feature_names = preprocessor_fitted.get_feature_names_out()

print("Number of transformed features:", len(feature_names))
print(feature_names[:10])

Number of transformed features: 367
['num__CNT_CHILDREN' 'num__AMT_INCOME_TOTAL' 'num__AMT_CREDIT'
 'num__AMT_ANNUITY' 'num__AMT_GOODS_PRICE'
 'num__REGION_POPULATION_RELATIVE' 'num__DAYS_REGISTRATION'
 'num__DAYS_ID_PUBLISH' 'num__OWN_CAR_AGE' 'num__FLAG_MOBIL']


**Step 3 — Calculate LightGBM Gain**

In [25]:

gain_values = lightgbm_model.booster_.feature_importance(
    importance_type='gain'
)

gain_df = pd.DataFrame({
    'Feature': feature_names,
    'Gain': gain_values
})

gain_df = (
    gain_df
    .sort_values('Gain', ascending=False)
    .reset_index(drop=True)
)

gain_df['Gain_percentage'] = (
    gain_df['Gain'] / gain_df['Gain'].sum()
) * 100

In [26]:
print("Shape", gain_df.shape)
gain_df.head(100)

Shape (367, 3)


,Feature,Gain,Gain_percentage
0,num__EXT_SOURCE_2,24140.187907,18.061509
1,num__EXT_SOURCE_3,23294.643782,17.428879
2,num__EXT_SOURCE_1,7196.996310,5.384739
3,num__ANNUITY_CREDIT_RATIO,5346.998801,4.000585
4,num__AGE,3199.928031,2.394162
5,num__POS_CASH_AVG_FUTURE_INST,3142.508003,2.351201
6,num__INSTALL_AVG_LOAN_LATE_RATE,3059.248718,2.288907
7,num__YEARS_EMPLOYED,2968.013835,2.220646
8,cat__CODE_GENDER_M,2315.676820,1.732572
9,num__AMT_ANNUITY,2105.719740,1.575484


**Step 4 — Calculate split importance too**

In [27]:
split_values = lightgbm_model.booster_.feature_importance(
    importance_type='split'
)

split_df = pd.DataFrame({
    'Feature': feature_names,
    'Split': split_values
})

split_df = (
    split_df
    .sort_values('Split', ascending=False)
    .reset_index(drop=True)
)

In [28]:
print("Shape", split_df.shape)
split_df.head(100)

Shape (367, 2)


,Feature,Split
0,num__EXT_SOURCE_1,135
1,num__ANNUITY_CREDIT_RATIO,124
2,num__EXT_SOURCE_2,102
3,num__EXT_SOURCE_3,97
4,num__AGE,92
5,num__POS_CASH_AVG_FUTURE_INST,84
6,num__AMT_ANNUITY,65
7,num__INSTALL_AVG_LOAN_LATE_RATE,64
8,num__YEARS_EMPLOYED,57
9,num__INSTALL_TOTAL_AMOUNT_PAID,55


**Step 5 — Put Gain and Split together**

In [29]:
importance_compare = pd.DataFrame({
    'Feature': gain_df['Feature'],
    'Gain': gain_df['Gain'],
    'Gain_percentage': gain_df['Gain_percentage'],
    'Split': split_df['Split']
})

importance_compare = (
    importance_compare
    .sort_values('Gain',ascending=False)
    .reset_index(drop=True)
)

In [30]:
importance_compare.head()

,Feature,Gain,Gain_percentage,Split
0,num__EXT_SOURCE_2,24140.187907,18.061509,135
1,num__EXT_SOURCE_3,23294.643782,17.428879,124
2,num__EXT_SOURCE_1,7196.996310,5.384739,102
3,num__ANNUITY_CREDIT_RATIO,5346.998801,4.000585,97
4,num__AGE,3199.928031,2.394162,92


**Step 6 — Find zero-Gain features**

In [31]:
zero_gain_features = gain_df[
    gain_df['Gain'] == 0
]['Feature'].tolist()

print("Zero-Gain Features:", len(zero_gain_features))

Zero-Gain Features: 129


**Step 7 — Now do correlation redundancy**

In [32]:
corr_matrix = X_train[num_features].corr().abs()

In [33]:
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

In [34]:
high_corr_pairs = (
    upper
    .stack()
    .reset_index()
)

high_corr_pairs.columns = [
    'Feature_1',
    'Feature_2',
    'Correlation'
]

high_corr_pairs = (
    high_corr_pairs[
        high_corr_pairs['Correlation'] >= 0.90
    ]
    .sort_values('Correlation', ascending=False)
    .reset_index(drop=True)
)

In [35]:
high_corr_pairs.shape

(95, 3)

In [36]:
high_corr_pairs.head()

,Feature_1,Feature_2,Correlation
0,PREV_APP_MAX_APPLICATION_AMOUNT,PREV_APP_MAX_GOODS_PRICE,0.999959
1,BUREAU_DAYS_CREDIT_MIN,BUREAU_BALANCE_MIN_MONTH,0.999915
2,BUREAU_TOTAL_LOANS,BUREAU_BALANCE_UNIQUE_CREDITS,0.999861
3,CC_MAX_CARD_DPD,CC_AVG_CARD_DPD,0.999329
4,CC_AVG_CARD_DPD_RATE,CC_MAX_CARD_DPD_RATE,0.999301


**Step 8 — Create a function to recover the original column**

In [37]:
def get_original_feature(feature_name, categorical_columns):

    # Numerical feature
    if feature_name.startswith('num__'):
        return feature_name.replace('num__', '', 1)

    # Categorical feature
    if feature_name.startswith('cat__'):
        encoded_name = feature_name.replace('cat__', '', 1)

        for col in categorical_columns:
            if encoded_name.startswith(col + '_'):
                return col

        return encoded_name

    return feature_name

**Step 9 — Apply the function**

In [38]:
gain_df['Original_Feature'] = gain_df['Feature'].apply(
    lambda x: get_original_feature(x, cat_features)
)

In [39]:
gain_df.head(2)

,Feature,Gain,Gain_percentage,Original_Feature
0,num__EXT_SOURCE_2,24140.187907,18.061509,EXT_SOURCE_2
1,num__EXT_SOURCE_3,23294.643782,17.428879,EXT_SOURCE_3


**Step 10 — ['original_gain'] =  Group by original feature and sum the Gain**

In [40]:
original_gain = (
    gain_df
    .groupby('Original_Feature', as_index=False)['Gain']
    .sum()
    .sort_values('Gain', ascending=False)
    .reset_index(drop=True)
)

**Step 11 — Add Gain percentage at original-column level**

In [41]:
original_gain['Gain_percentage'] = (
    original_gain['Gain'] /
    original_gain['Gain'].sum()
) * 100

In [42]:
print("ORIGNAL GAIN SHAPE : ",original_gain.shape)
original_gain.head(2)

ORIGNAL GAIN SHAPE :  (260, 3)


,Original_Feature,Gain,Gain_percentage
0,EXT_SOURCE_2,24140.187907,18.061509
1,EXT_SOURCE_3,23294.643782,17.428879


**Step 12 — Compare the original Gain with your correlation table**

In [43]:
gain_lookup = (
    original_gain
    .set_index('Original_Feature')['Gain']
    .to_dict()
)

**Step 13 — Add Gain of Feature 1 and Feature 2**

In [44]:
high_corr_pairs['Gain_1'] = (
    high_corr_pairs['Feature_1']
    .map(gain_lookup)
)

high_corr_pairs['Gain_2'] = (
    high_corr_pairs['Feature_2']
    .map(gain_lookup)
)

**Step 14 — Automatically suggest which correlated feature is weaker**

In [45]:
high_corr_pairs['Keep'] = np.where(
    high_corr_pairs['Gain_1'] >= high_corr_pairs['Gain_2'],
    high_corr_pairs['Feature_1'],
    high_corr_pairs['Feature_2']
)

high_corr_pairs['Suggested_Remove'] = np.where(
    high_corr_pairs['Gain_1'] >= high_corr_pairs['Gain_2'],
    high_corr_pairs['Feature_2'],
    high_corr_pairs['Feature_1']
)

In [46]:
print("SHAPE: ",high_corr_pairs.shape)
high_corr_pairs.head()

SHAPE:  (95, 7)


,Feature_1,Feature_2,Correlation,Gain_1,Gain_2,Keep,Suggested_Remove
0,PREV_APP_MAX_APPLICATION_AMOUNT,PREV_APP_MAX_GOODS_PRICE,0.999959,179.306998,16.854300,PREV_APP_MAX_APPLICATION_AMOUNT,PREV_APP_MAX_GOODS_PRICE
1,BUREAU_DAYS_CREDIT_MIN,BUREAU_BALANCE_MIN_MONTH,0.999915,324.960799,113.003209,BUREAU_DAYS_CREDIT_MIN,BUREAU_BALANCE_MIN_MONTH
2,BUREAU_TOTAL_LOANS,BUREAU_BALANCE_UNIQUE_CREDITS,0.999861,72.643200,16.657101,BUREAU_TOTAL_LOANS,BUREAU_BALANCE_UNIQUE_CREDITS
3,CC_MAX_CARD_DPD,CC_AVG_CARD_DPD,0.999329,57.882299,29.073701,CC_MAX_CARD_DPD,CC_AVG_CARD_DPD
4,CC_AVG_CARD_DPD_RATE,CC_MAX_CARD_DPD_RATE,0.999301,41.900002,33.618100,CC_AVG_CARD_DPD_RATE,CC_MAX_CARD_DPD_RATE


### **REMOVING THE FEATURE**

**Step 1 — Find original columns with zero total Gain**

In [47]:
zero_gain_original = original_gain[
    original_gain['Gain'] == 0
].copy()


print(
    "Original columns with zero total Gain:",
    len(zero_gain_original)
)

zero_gain_original.head(60)

Original columns with zero total Gain: 52


,Original_Feature,Gain,Gain_percentage
208,CC_TOTAL_OTHER_DRAWINGS,0.0,0.0
209,CNT_CHILDREN,0.0,0.0
210,ELEVATORS_AVG,0.0,0.0
211,ELEVATORS_MEDI,0.0,0.0
212,CC_STATUS_SENT_PROPOSAL_COUNT,0.0,0.0
213,CC_STATUS_REFUSED_COUNT,0.0,0.0
214,CC_STATUS_SIGNED_COUNT,0.0,0.0
215,CC_TOTAL_OTHER_DRAWING_COUNT,0.0,0.0
216,CC_STATUS_APPROVED_COUNT,0.0,0.0
217,CC_STATUS_DEMAND_COUNT,0.0,0.0


**Step 2 — Create the ≥0.95 correlation dataframe**

In [48]:
very_high_corr = high_corr_pairs[
    high_corr_pairs['Correlation'] >= 0.95
].copy()

print("Highly correlated pairs:", len(very_high_corr))

Highly correlated pairs: 67


In [49]:
very_high_corr.sort_values(
    'Correlation', ascending=False
).head(70)

,Feature_1,Feature_2,Correlation,Gain_1,Gain_2,Keep,Suggested_Remove
0,PREV_APP_MAX_APPLICATION_AMOUNT,PREV_APP_MAX_GOODS_PRICE,0.999959,179.306998,16.854300,PREV_APP_MAX_APPLICATION_AMOUNT,PREV_APP_MAX_GOODS_PRICE
1,BUREAU_DAYS_CREDIT_MIN,BUREAU_BALANCE_MIN_MONTH,0.999915,324.960799,113.003209,BUREAU_DAYS_CREDIT_MIN,BUREAU_BALANCE_MIN_MONTH
2,BUREAU_TOTAL_LOANS,BUREAU_BALANCE_UNIQUE_CREDITS,0.999861,72.643200,16.657101,BUREAU_TOTAL_LOANS,BUREAU_BALANCE_UNIQUE_CREDITS
3,CC_MAX_CARD_DPD,CC_AVG_CARD_DPD,0.999329,57.882299,29.073701,CC_MAX_CARD_DPD,CC_AVG_CARD_DPD
4,CC_AVG_CARD_DPD_RATE,CC_MAX_CARD_DPD_RATE,0.999301,41.900002,33.618100,CC_AVG_CARD_DPD_RATE,CC_MAX_CARD_DPD_RATE
5,YEARS_BUILD_AVG,YEARS_BUILD_MEDI,0.998428,0.000000,27.010800,YEARS_BUILD_MEDI,YEARS_BUILD_AVG
6,OBS_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,0.998386,110.596599,87.588699,OBS_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE
7,FLOORSMIN_AVG,FLOORSMIN_MEDI,0.997187,82.471300,0.000000,FLOORSMIN_AVG,FLOORSMIN_MEDI
8,CC_AVG_CARD_MONTHS,CC_MAX_CARD_MONTHS,0.997173,360.359004,89.832699,CC_AVG_CARD_MONTHS,CC_MAX_CARD_MONTHS
9,FLOORSMAX_AVG,FLOORSMAX_MEDI,0.997022,47.337698,32.916800,FLOORSMAX_AVG,FLOORSMAX_MEDI


In [50]:
redundant_candidates = [
    'PREV_APP_MAX_GOODS_PRICE',
    'BUREAU_BALANCE_MIN_MONTH',
    'BUREAU_BALANCE_UNIQUE_CREDITS',
    'CC_AVG_CARD_DPD',
    'CC_MAX_CARD_DPD_RATE',
    'OBS_60_CNT_SOCIAL_CIRCLE',
    'CC_MAX_CARD_MONTHS',
    'FLOORSMAX_MEDI',
    'ENTRANCES_AVG',
    'COMMONAREA_MEDI',
    'LIVINGAREA_AVG',
    'APARTMENTS_MEDI',
    'BASEMENTAREA_MEDI',
    'LIVINGAPARTMENTS_MEDI',
    'LANDAREA_MEDI',
    'YEARS_BUILD_MEDI',
    'FLOORSMIN_MODE',
    'ELEVATORS_MODE',
    'NONLIVINGAREA_MEDI',
    'NONLIVINGAPARTMENTS_MEDI',
    'ELEVATORS_AVG',
    'COMMONAREA_MODE',
    'APARTMENTS_MODE',
    'LIVINGAREA_MODE',
    'BASEMENTAREA_MODE',
    'NONLIVINGAREA_MODE',
    'LIVINGAPARTMENTS_AVG',
    'NONLIVINGAREA_AVG',
    'YEARS_BEGINEXPLUATATION_AVG',
    'YEARS_BEGINEXPLUATATION_MEDI',
    'INSTALL_LOANS',
    'PREV_APP_PORTFOLIO_POS_COUNT',
    'CC_TOTAL_POS_DRAWING_COUNT',
    'POS_CASH_AVG_INST'
]

**Step 3 — Removal Candidates Set**

In [51]:
# Step 1: all original columns with zero total Gain
features_to_remove = set(
    zero_gain_original['Original_Feature']
)

# Step 2: manually reviewed redundancy candidates
features_to_remove.update(redundant_candidates)

print("Total features to remove:", len(features_to_remove))

Total features to remove: 82


**Step 4 - Selected Features**

In [52]:
selected_features = [
    col for col in X_train.columns
    if col not in features_to_remove
]

X_train_selected = X_train[selected_features].copy()
X_val_selected = X_val[selected_features].copy()

print("Original features:", X_train.shape[1])
print("Removed features:", len(features_to_remove))
print("Selected features:", len(selected_features))

Original features: 260
Removed features: 82
Selected features: 178


### **Recreating the Baseline model after the Feature Selection**

**Step 1 - Preprocessing Pipeline**

In [53]:
# Create the numerical and categorical lists again:

num_features_selected = X_train_selected.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

cat_features_selected = X_train_selected.select_dtypes(
    include=['object', 'category']
).columns.tolist()

print("Numerical:", len(num_features_selected))
print("Categorical:", len(cat_features_selected))

Numerical: 164
Categorical: 14


In [54]:
# Transformers

numeric_transformer_selected = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer_selected = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        handle_unknown='ignore',
        drop='first',
        sparse_output=False
    ))
])

preprocessor_selected = ColumnTransformer([
    ('num', numeric_transformer_selected, num_features_selected),
    ('cat', categorical_transformer_selected, cat_features_selected)
])

**Step 2 - Baseline Models**

In [55]:
models_selected = {
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    ),

    'XGBoost': XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1
    ),

    'LightGBM': LGBMClassifier(
        n_estimators=100,
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    )
}

In [56]:
trained_models_selected = {}
results_selected = []

for name, model in models_selected.items():

    print(f"Training {name}...")

    pipeline = Pipeline([
        ('preprocessor', preprocessor_selected),
        ('model', model)
    ])

    pipeline.fit(X_train_selected, y_train)

    trained_models_selected[name] = pipeline

    y_pred = pipeline.predict(X_val_selected)
    y_prob = pipeline.predict_proba(X_val_selected)[:, 1]

    results_selected.append({
        'Model': name,
        'ROC-AUC': roc_auc_score(y_val, y_prob),
        'Recall': recall_score(y_val, y_pred),
        'Precision': precision_score(y_val, y_pred),
        'F1-Score': f1_score(y_val, y_pred)
    })

results_selected_df = (
    pd.DataFrame(results_selected)
    .sort_values('ROC-AUC', ascending=False)
    .reset_index(drop=True)
)

display(results_selected_df)

Training Random Forest...
Training XGBoost...
Training LightGBM...


,Model,ROC-AUC,Recall,Precision,F1-Score
0,LightGBM,0.785531,0.040548,0.557196,0.075594
1,XGBoost,0.770211,0.067669,0.475472,0.118477
2,Random Forest,0.744475,0.002148,0.333333,0.004269


**Step 3 - compare before vs after feature selection**

In [57]:
baseline_before = results_df[
    ['Model', 'ROC-AUC', 'Recall']
].copy()

baseline_before = baseline_before.rename(columns={
    'ROC-AUC': 'ROC-AUC_Before',
    'Recall': 'Recall_Before'
})

baseline_after = results_selected_df[
    ['Model', 'ROC-AUC', 'Recall']
].copy()

baseline_after = baseline_after.rename(columns={
    'ROC-AUC': 'ROC-AUC_After',
    'Recall': 'Recall_After'
})

comparison = baseline_before.merge(
    baseline_after,
    on='Model'
)

comparison['ROC-AUC_Change'] = (
    comparison['ROC-AUC_After']
    - comparison['ROC-AUC_Before']
)

comparison['Recall_Change'] = (
    comparison['Recall_After']
    - comparison['Recall_Before']
)

comparison

,Model,ROC-AUC_Before,Recall_Before,ROC-AUC_After,Recall_After,ROC-AUC_Change,Recall_Change
0,LightGBM,0.784547,0.040279,0.785531,0.040548,0.000984,0.000269
1,XGBoost,0.769887,0.066058,0.770211,0.067669,0.000323,0.001611
2,Random Forest,0.741291,0.002148,0.744475,0.002148,0.003183,0.000000


### **Hyperparameter Tuning**

**1. LightGBM Model**

In [58]:
# LightGBM Pipeline

lgbm_pipeline = Pipeline([
    ('preprocessor', preprocessor_selected),
    ('model', LGBMClassifier(
        objective='binary',
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

In [59]:
# LightGBM search space

lgbm_param_dist = {
    'model__n_estimators': [100, 200, 300, 500],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
    'model__num_leaves': [15, 31, 50, 75],
    'model__max_depth': [-1, 5, 7, 10],
    'model__min_child_samples': [20, 50, 100, 200],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'model__reg_alpha': [0, 0.1, 0.5, 1.0],
    'model__reg_lambda': [0, 0.1, 0.5, 1.0]
}

In [60]:
# RandomizedSearchCV for LightGBM

lgbm_search = RandomizedSearchCV(
    estimator=lgbm_pipeline,
    param_distributions=lgbm_param_dist,
    n_iter=10,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

lgbm_search.fit(X_train_selected, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median'))]),
                                                                               ['AMT_INCOME_TOTAL',
                                                                                'AMT_CREDIT',
                                                                                'AMT_ANNUITY',
                                                                                'AMT_GOODS_PRICE',
                                                                                'REGION_POPULATION_RELATIVE',
                                                                                'DAYS_REGISTRATION',
                                                                                'DAYS_ID_PUBLISH',
                                                                                'OWN_CAR_AGE',
                                                                                'FLAG_WORK_PHONE',
                                                                                'FLAG_CONT_MOBILE'...
                                                                    0.9, 1.0],
                                        'model__learning_rate': [0.01, 0.03,
                                                                 0.05, 0.1],
                                        'model__max_depth': [-1, 5, 7, 10],
                                        'model__min_child_samples': [20, 50,
                                                                     100, 200],
                                        'model__n_estimators': [100, 200, 300,
                                                                500],
                                        'model__num_leaves': [15, 31, 50, 75],
                                        'model__reg_alpha': [0, 0.1, 0.5, 1.0],
                                        'model__reg_lambda': [0, 0.1, 0.5, 1.0],
                                        'model__subsample': [0.7, 0.8, 0.9,
                                                             1.0]},
                   random_state=42, scoring='roc_auc', verbose=1)

In [61]:
print("Best LightGBM ROC-AUC:", lgbm_search.best_score_)
print("Best LightGBM parameters:")
print(lgbm_search.best_params_)

Best LightGBM ROC-AUC: 0.7811512943090649
Best LightGBM parameters:
{'model__subsample': 0.7, 'model__reg_lambda': 1.0, 'model__reg_alpha': 1.0, 'model__num_leaves': 50, 'model__n_estimators': 300, 'model__min_child_samples': 200, 'model__max_depth': 7, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.8}


In [62]:
# Best lgbm train model
lgbm_best = lgbm_search.best_estimator_

In [63]:
y_lgbm_pred = lgbm_best.predict(X_val_selected)
y_lgbm_prob = lgbm_best.predict_proba(X_val_selected)[:, 1]

In [64]:
recall_lgbm = recall_score(y_val, y_lgbm_pred)
precision_lgbm = precision_score(y_val, y_lgbm_pred)
f1_lgbm = f1_score(y_val, y_lgbm_pred)
roc_auc_lgbm = roc_auc_score(y_val, y_lgbm_prob)

print("LightGBM Recall:", recall_lgbm)
print("LightGBM Precision:", precision_lgbm)
print("LightGBM F1-Score:", f1_lgbm)
print("LightGBM ROC-AUC:", roc_auc_lgbm)

LightGBM Recall: 0.04833512352309345
LightGBM Precision: 0.6
LightGBM F1-Score: 0.08946322067594434
LightGBM ROC-AUC: 0.7891650810302151


In [78]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_lgbm_pred))


Confusion Matrix:
[[42282   120]
 [ 3544   180]]


In [65]:
print(classification_report(y_val, y_lgbm_pred))

              precision    recall  f1-score   support

           0       0.92      1.00      0.96     42402
           1       0.60      0.05      0.09      3724

    accuracy                           0.92     46126
   macro avg       0.76      0.52      0.52     46126
weighted avg       0.90      0.92      0.89     46126



**2. XGBoost Model**

In [66]:
# XGBoost Pipeline

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor_selected),
    ('model', XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ))
])

In [67]:
# XGBoost search space

xgb_param_dist = {
    'model__n_estimators': [100, 200, 300, 500],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
    'model__max_depth': [3, 4, 5, 6, 8],
    'model__min_child_weight': [1, 3, 5, 10],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'model__gamma': [0, 0.1, 0.3, 0.5],
    'model__reg_alpha': [0, 0.1, 0.5, 1.0],
    'model__reg_lambda': [1, 2, 5, 10]
}

In [68]:
# RandomizedSearchCV for XGBoost

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=10,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train_selected, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median'))]),
                                                                               ['AMT_INCOME_TOTAL',
                                                                                'AMT_CREDIT',
                                                                                'AMT_ANNUITY',
                                                                                'AMT_GOODS_PRICE',
                                                                                'REGION_POPULATION_RELATIVE',
                                                                                'DAYS_REGISTRATION',
                                                                                'DAYS_ID_PUBLISH',
                                                                                'OWN_CAR_AGE',
                                                                                'FLAG_WORK_PHONE',
                                                                                'FLAG_CONT_MOBILE'...
                   param_distributions={'model__colsample_bytree': [0.7, 0.8,
                                                                    0.9, 1.0],
                                        'model__gamma': [0, 0.1, 0.3, 0.5],
                                        'model__learning_rate': [0.01, 0.03,
                                                                 0.05, 0.1],
                                        'model__max_depth': [3, 4, 5, 6, 8],
                                        'model__min_child_weight': [1, 3, 5,
                                                                    10],
                                        'model__n_estimators': [100, 200, 300,
                                                                500],
                                        'model__reg_alpha': [0, 0.1, 0.5, 1.0],
                                        'model__reg_lambda': [1, 2, 5, 10],
                                        'model__subsample': [0.7, 0.8, 0.9,
                                                             1.0]},
                   random_state=42, scoring='roc_auc', verbose=1)

In [69]:
print("Best XGBoost ROC-AUC:", xgb_search.best_score_)
print("Best XGBoost parameters:")
print(xgb_search.best_params_)

Best XGBoost ROC-AUC: 0.7796879607667346
Best XGBoost parameters:
{'model__subsample': 0.8, 'model__reg_lambda': 5, 'model__reg_alpha': 1.0, 'model__n_estimators': 200, 'model__min_child_weight': 1, 'model__max_depth': 8, 'model__learning_rate': 0.05, 'model__gamma': 0.3, 'model__colsample_bytree': 0.8}


In [70]:
# Best xgb train model
xgb_best = xgb_search.best_estimator_

In [71]:
y_xgb_pred = xgb_best.predict(X_val_selected)
y_xgb_prob = xgb_best.predict_proba(X_val_selected)[:, 1]

In [72]:
recall_xgb = recall_score(y_val, y_xgb_pred)
precision_xgb = precision_score(y_val, y_xgb_pred)
f1_xgb = f1_score(y_val, y_xgb_pred)
roc_auc_xgb = roc_auc_score(y_val, y_xgb_prob)

print("XGBoost Recall:", recall_xgb)
print("XGBoost Precision:", precision_xgb)
print("XGBoost F1-Score:", f1_xgb)
print("XGBoost ROC-AUC:", roc_auc_xgb)

XGBoost Recall: 0.04054779806659506
XGBoost Precision: 0.5992063492063492
XGBoost F1-Score: 0.07595573440643863
XGBoost ROC-AUC: 0.7869902867196494


In [77]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_xgb_pred))


Confusion Matrix:
[[42301   101]
 [ 3573   151]]


In [73]:
print(classification_report(y_val, y_xgb_pred))

              precision    recall  f1-score   support

           0       0.92      1.00      0.96     42402
           1       0.60      0.04      0.08      3724

    accuracy                           0.92     46126
   macro avg       0.76      0.52      0.52     46126
weighted avg       0.90      0.92      0.89     46126



## **Check Thresholds**

In [74]:
thresholds = np.arange(0.10, 0.51, 0.05)

threshold_results = []

for threshold in thresholds:

    y_lgbm_pred_threshold = (
        y_lgbm_prob >= threshold
    ).astype(int)

    threshold_results.append({
        'Threshold': threshold,
        'Recall': recall_score(
            y_val,
            y_lgbm_pred_threshold,
            zero_division=0
        ),
        'Precision': precision_score(
            y_val,
            y_lgbm_pred_threshold,
            zero_division=0
        ),
        'F1-Score': f1_score(
            y_val,
            y_lgbm_pred_threshold,
            zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df)

,Threshold,Recall,Precision,F1-Score
0,0.10,0.632922,0.210465,0.315888
1,0.15,0.476369,0.268462,0.343399
2,0.20,0.350698,0.315307,0.332062
3,0.25,0.260473,0.363977,0.303647
4,0.30,0.186627,0.398967,0.254299
5,0.35,0.136950,0.442708,0.209188
6,0.40,0.097744,0.480845,0.162464
7,0.45,0.070086,0.526210,0.123697
8,0.50,0.048335,0.600000,0.089463


* The classification threshold was reduced from 0.50 to 0.15 to improve default detection. This increased Recall substantially, while maintaining a better balance between Recall and Precision, as reflected by the highest F1-score among the tested thresholds.

* I would not choose 0.10 just because Recall is highest. It sacrifices too much Precision compared with 0.15.

# **Final Test Result**

**Evaluate the already-tuned LightGBM + Threshold 0.15 on the untouched test set.**

In [75]:
# Final test probabilities
y_test_prob = lgbm_best.predict_proba(
    X_test[selected_features]
)[:, 1]

# Apply selected validation threshold
final_threshold = 0.15

y_test_pred = (
    y_test_prob >= final_threshold
).astype(int)

In [76]:
print("Final Test ROC-AUC:",
      roc_auc_score(y_test, y_test_prob))

print("Final Test Recall:",
      recall_score(y_test, y_test_pred))

print("Final Test Precision:",
      precision_score(y_test, y_test_pred))

print("Final Test F1:",
      f1_score(y_test, y_test_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Final Test ROC-AUC: 0.7866257360294081
Final Test Recall: 0.4675080558539205
Final Test Precision: 0.2653963414634146
Final Test F1: 0.33858420847919096

Confusion Matrix:
[[37584  4819]
 [ 1983  1741]]


## **Final Model Evaluation**

After feature selection and hyperparameter tuning, LightGBM was selected as the final model based on its validation ROC-AUC performance. The classification threshold was then optimized on the validation set to improve default detection, and a threshold of **0.15** was selected based on the Recall–Precision trade-off and the highest F1-score among the evaluated thresholds.

The final model was evaluated once on the untouched test set.

| Metric    | Test Performance |
| --------- | ---------------: |
| ROC-AUC   |       **0.7866** |
| Recall    |       **46.75%** |
| Precision |       **26.54%** |
| F1-Score  |       **33.86%** |

The confusion matrix was:

* True Negatives: **37,584**
* False Positives: **4,819**
* False Negatives: **1,983**
* True Positives: **1,741**

The final ROC-AUC of **0.7866** indicates that the model provides useful ranking ability for distinguishing defaulters from non-defaulters. Using the optimized threshold of **0.15** substantially improved Recall compared with the default 0.50 threshold, allowing the model to identify a larger proportion of actual defaulters while accepting a reduction in Precision.

Overall, the final model prioritizes **default detection and Recall** rather than maximizing Precision alone, which is appropriate for this credit-risk use case.


## **Save the final model**

In [80]:
final_model_artifact = {
    "model": lgbm_best,
    "selected_features": selected_features,
    "threshold": 0.15
}

joblib.dump(
    final_model_artifact,
    "home_credit_final_model.pkl"
)

['home_credit_final_model.pkl']